In [ ]:
# @title Setup: local library, Google Drive, or GitHub
import os
import subprocess
import sys
from pathlib import Path

USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
USE_GITHUB = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/Baoshan-Song/KFV-FGO-Comparison.git"
BRANCH = "python_colab"
PROJECT_NAME = "KFV-FGO-Comparison"

DRIVE_ROOT = Path("/content/drive/MyDrive")
if USE_GOOGLE_DRIVE:
  try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted:", DRIVE_ROOT.exists())
  except ImportError:
    print("No Colab runtime; Google Drive mount skipped.")

if USE_GITHUB:
  root = Path("/content") if Path("/content").exists() else Path.cwd().parent
  PROJECT_PATH = root / PROJECT_NAME
  if not PROJECT_PATH.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL,
                    str(PROJECT_PATH)], check=True)
else:
  candidates = [
      DRIVE_ROOT / "Colab Notebooks" / "KFV-FGO-Comparison-main",
      Path.cwd(),
      Path.home() / "Library/CloudStorage/GoogleDrive-zsyqdl@gmail.com/我的云端硬盘/Colab Notebooks/KFV-FGO-Comparison-main",
  ]
  PROJECT_PATH = next(
      (path for path in candidates if (path / "experiment_common.py").exists()),
      None,
  )
  if PROJECT_PATH is None:
    raise FileNotFoundError("Cannot locate the local KFV-FGO project")

os.chdir(PROJECT_PATH)
if str(PROJECT_PATH) not in sys.path:
  sys.path.insert(0, str(PROJECT_PATH))
print(f"Using project: {PROJECT_PATH}")

In [ ]:
# @title 1. KFV vs KFV
# @markdown Simulation parameters and both KFV estimator parameters are adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}  # Unit: steps
distance = 105  # @param {type:"slider", min:200, max:1000, step:50}  # Unit: m
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.  # @param {type:"slider", min:0, max:1, step:0.05}  # Ratio (0-1)
outlier_mean = 0.0  # @param {type:"number"}  # Unit: m
outlier_sigma = 10.0  # @param {type:"number"}  # Unit: m
white_sigma = 0.1  # @param {type:"number"}  # Unit: m

# --------------------------------------------------
# Left Column KFV Configuration
# --------------------------------------------------
left_mode = "EKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]
left_iterations = 2  # @param {type:"integer", min:1, max:20} # [Note] Only used in iterative modes: iEKF, riEKF
left_kernel = "huber"  # @param ["none", "huber"] # [Note] Only used in robust modes: rEKF, riEKF
left_delta = 2.0  # @param {type:"number"} # [Note] Only used in robust modes: rEKF, riEKF
# --------------------------------------------------
# Right Column KFV Configuration
# --------------------------------------------------
right_mode = "riEKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]
right_iterations = 20  # @param {type:"integer", min:1, max:20} # [Note] Only used in iterative modes: iEKF, riEKF
right_kernel = "huber"  # @param ["none", "huber"] # [Note] Only used in robust modes: rEKF, riEKF
right_delta = 2.0  # @param {type:"number"} # [Note] Only used in robust modes: rEKF, riEKF
# --------------------------------------------------
# Helper Functions and Execution
# --------------------------------------------------
def make_ui_config(mode, iterations, kernel, delta):
  return make_config(kfv_mode=mode, max_iteration=iterations, robust_kernel=kernel, robust_delta=delta)

sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

left = {"name": f"KFV-{left_mode}", "kind": "kfv", "config": make_ui_config(
    left_mode, left_iterations, left_kernel, left_delta)}

right = {"name": f"KFV-{right_mode}", "kind": "kfv", "config": make_ui_config(
    right_mode, right_iterations, right_kernel, right_delta)}

experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))

In [ ]:
# @title 2. KFV vs FGO
# @markdown Simulation parameters, KFV parameters, and FGO parameters are all adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}  # Unit: steps
distance = 105  # @param {type:"slider", min:200, max:1000, step:50}  # Unit: m
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.35  # @param {type:"slider", min:0, max:1, step:0.05}  # Ratio (0-1)
outlier_mean = 30.0  # @param {type:"number"}  # Unit: m
outlier_sigma = 5.0  # @param {type:"number"}  # Unit: m
white_sigma = 0.1  # @param {type:"number"}  # Unit: m

# --------------------------------------------------
# Left Column KFV Configuration
# --------------------------------------------------
kfv_mode = "iEKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]
# fmt: off
kfv_iterations = 10  # @param {type:"integer", min:1, max:20} # [Note] Only used in iterative modes: iEKF, riEKF
kfv_kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"] # [Note] Only used in robust modes: rEKF, riEKF
kfv_delta = 2.0  # @param {type:"number"} # [Note] Only used in robust modes: rEKF, riEKF
# fmt: on

# --------------------------------------------------
# Right Column FGO Configuration
# --------------------------------------------------
# fmt: off
fgo_iterations = 10  # @param {type:"integer", min:1, max:20} # [Note] Only used in iterative modes: iEKF, riEKF (if imitate_kfv=True)
fgo_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"] # [Note] Only used in robust modes: rEKF, riEKF (if imitate_kfv=True)
fgo_delta = 2.0  # @param {type:"number"} # [Note] Only used in robust modes: rEKF, riEKF (if imitate_kfv=True)
fgo_imitate_kfv = True  # @param {type:"boolean"} # [Note] Overrides window_size to 1 and syncs iteration/kernel logic with KFV mode
fgo_window_size = 1  # @param {type:"integer", min:1, max:20} # [Note] Forced to 1 if imitate_kfv=True; must be > 1 if imitate_kfv=False
fgo_auto_diff = False  # @param {type:"boolean"}
# fmt: on

# --------------------------------------------------
# Hierarchical Parameter Mapping for imitate_kfv
# --------------------------------------------------
def make_hierarchical_fgo_config(target_kfv_mode, raw_iterations, raw_kernel, raw_delta,
                                imitate, window, autodiff):
  """
  Applies hierarchical parameter mapping according to the specified KFV mode.
  """
  if imitate:
    window = 1  # Force window size to 1 in imitation mode
    if target_kfv_mode == "EKF":
      iterations = 1
      kernel = "none"
    elif target_kfv_mode == "iEKF":
      iterations = raw_iterations
      kernel = "none"
    elif target_kfv_mode == "rEKF":
      iterations = 1
      kernel = raw_kernel
    elif target_kfv_mode == "riEKF":
      iterations = raw_iterations
      kernel = raw_kernel
    else:
      iterations = raw_iterations
      kernel = raw_kernel
  else:
    iterations = raw_iterations
    kernel = raw_kernel

  return make_config(
      kfv_mode=target_kfv_mode,
      max_iteration=iterations,
      robust_kernel=kernel,
      robust_delta=raw_delta,
      imitate_kfv=imitate,
      window_size=window,
      autodiff=autodiff
  )

# --------------------------------------------------
# Simulation & Execution
# --------------------------------------------------
sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

# Left Estimator (KFV)
left_config = make_config(
    kfv_mode=kfv_mode,
    max_iteration=kfv_iterations,
    robust_kernel=kfv_kernel,
    robust_delta=kfv_delta
)

# Right Estimator (Re-FGO) - Uses hierarchical argument mapping
right_config = make_hierarchical_fgo_config(
    target_kfv_mode=kfv_mode,
    raw_iterations=fgo_iterations,
    raw_kernel=fgo_kernel,
    raw_delta=fgo_delta,
    imitate=fgo_imitate_kfv,
    window=fgo_window_size,
    autodiff=fgo_auto_diff
)

left = {"name": f"KFV-{kfv_mode}", "kind": "kfv", "config": left_config}
right = {"name": "Re-FGO", "kind": "fgo", "config": right_config}

experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))

In [ ]:
# @title 3. FGO vs FGO
# @markdown Simulation parameters and both FGO configurations are adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}  # Unit: steps
distance = 105  # @param {type:"slider", min:200, max:1000, step:50}  # Unit: m
seed = 7  # @param {type:"integer", min:0, max:999} # random seed
outlier_weight = 0.2  # @param {type:"slider", min:0, max:1, step:0.05}  # Ratio (0-1)
outlier_mean = 30.0  # @param {type:"number"}  # Unit: m
outlier_sigma = 5.0  # @param {type:"number"}  # Unit: m
white_sigma = 0.1  # @param {type:"number"}  # Unit: m

# --------------------------------------------------
# Left Column FGO Configuration
# --------------------------------------------------
left_iterations = 2  # @param {type:"integer", min:1, max:20}
left_kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"]
left_delta = 2.0  # @param {type:"number"}
left_imitate_kfv = False  # @param {type:"boolean"}
left_window_size = 2  # @param {type:"integer", min:1, max:20} # [Note] window_size must be > 1 when imitate_kfv=False
left_auto_diff = False  # @param {type:"boolean"}

# --------------------------------------------------
# Right Column FGO Configuration
# --------------------------------------------------
right_iterations = 2  # @param {type:"integer", min:1, max:20}
right_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"]
right_delta = 2.0  # @param {type:"number"}
right_imitate_kfv = False  # @param {type:"boolean"}
right_window_size = 20  # @param {type:"integer", min:1, max:20} # [Note] window_size must be > 1 when imitate_kfv=False
right_auto_diff = False  # @param {type:"boolean"}

# --------------------------------------------------
# Helper Functions and Execution
# --------------------------------------------------
def make_ui_config(iterations, kernel, delta, imitate, window, autodiff):
  return make_config(
      err_x=100.0, err_y=-100.0, err_vx=0.0, err_vy=0.0,
      p0_diag=(50.0, 50.0, 1.0, 1.0), kfv_mode="FGO",
      robust_kernel=kernel, robust_delta=delta, max_iteration=iterations,
      window_size=window, imitate_kfv=imitate, autodiff=autodiff
  )

sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

left = {"name": "FGO-config-1", "kind": "fgo", "config": make_ui_config(
    left_iterations, left_kernel, left_delta, left_imitate_kfv, left_window_size, left_auto_diff)}

right = {"name": "FGO-config-2", "kind": "fgo", "config": make_ui_config(
    right_iterations, right_kernel, right_delta, right_imitate_kfv, right_window_size, right_auto_diff)}

experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))